# 📻 FM 음성 변조 실습 — 내 목소리를 주파수에 싣기

AM 실습에서는 내 목소리를 **반송파의 진폭(크기)** 에 실었습니다.
이번에는 같은 목소리를 **반송파의 주파수** 에 실어 보냅니다. 이것이 FM 라디오(88~108 MHz)의 원리입니다.

| | AM (진폭 변조) | FM (주파수 변조) |
|---|---|---|
| 음성이 바꾸는 것 | 반송파의 **크기** | 반송파의 **주파수(빠르기)** |
| 변조된 파형 | 위아래로 부풀었다 줄었다 | 크기는 일정, 촘촘했다 성겼다 |
| 잡음에 대한 강도 | 약함 (잡음이 곧 크기 변화) | 강함 (크기 잡음은 무시) |
| 필요한 대역폭 | 2 × 음성 대역 | 2 × (최대 주파수 편이 + 음성 대역) — 더 넓음 |

### 실습 순서
0. 준비 — 라이브러리, 한글 폰트, 파라미터
1. 음성 입력 — 마이크 녹음 또는 파일 업로드
2. 신호파 만들기 — 음성 대역(300~3400 Hz)만 남기기
3. **FM 변조** — 적분 → 위상 → 반송파
4. 송신 — 소리로 들어 보기, 잡음 채널 통과
5. **수신·복조** — 순간 주파수 측정으로 음성 복원
6. 실험 — ① 주파수 편이 Δf 바꾸기 ② AM과 FM의 잡음 비교
7. 생각해 볼 문제 + 파이썬 문법 정리

### 이 노트북에서 함께 익히는 파이썬 문법
각 단계 아래에 **🐍 파이썬 문법 노트** 가 있습니다. 코드를 실행한 다음 노트를 읽고, 코드에서 그 부분을 직접 찾아보세요.

`import / as` · 변수와 상수 · 함수 `def` · 기본값 인자 · `return` 여러 개 · 독스트링 · f-string · `if / elif / else` · `with open` · 리스트 · 딕셔너리 · `for` / `enumerate` / `zip` · 리스트 컴프리헨션 · NumPy 배열 연산 · 슬라이싱 · 불리언 마스크

> ▶ 셀은 **위에서부터 순서대로** 실행하세요 (`Shift + Enter`). 중간 셀만 실행하면 앞에서 만든 변수가 없어서 오류가 납니다.

## 0. 준비 — 라이브러리 · 한글 폰트 · 파라미터

In [ ]:
# ── 라이브러리 불러오기 ─────────────────────────────
import numpy as np                      # 숫자 배열 계산 (np 라는 짧은 이름으로 사용)
import matplotlib.pyplot as plt         # 그래프 그리기
import matplotlib.font_manager as fm    # 글꼴 관리
from scipy import signal                # 필터, 스펙트럼, 힐베르트 변환
from scipy.io import wavfile            # WAV 파일 읽기/쓰기
import subprocess, base64               # 외부 프로그램 실행, 녹음 데이터 해독
from IPython.display import Audio, Javascript, display   # 노트북에서 소리 재생, JS 실행
from google.colab import output, files  # Colab 전용: 브라우저와 통신, 파일 업로드

# ── 한글 폰트 설치 (그래프 제목의 한글이 □□로 깨지지 않게) ──
subprocess.run('apt-get -qq install -y fonts-nanum > /dev/null 2>&1', shell=True)
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)     # 음수 기호(-)가 깨지지 않게

# ── 실습 파라미터 (대문자 = 실습 내내 바꾸지 않을 '상수'라는 약속) ──
FS      = 48000    # 샘플링 주파수 [Hz] : 1초에 숫자 48000개로 소리를 표현
FC      = 10000    # 반송파 주파수 [Hz] : 귀에 '삐-' 하고 들리는 높이
F_LOW   = 300      # 음성 대역 하한 [Hz] (전화 음성 기준)
F_HIGH  = 3400     # 음성 대역 상한 [Hz]
DELTA_F = 3000     # 최대 주파수 편이 Δf [Hz] : 목소리가 가장 클 때 반송파가 벗어나는 폭
SNR_DB  = 20       # 채널의 신호 대 잡음비 [dB] : 작을수록 잡음이 큼

print('준비 완료!')
print(f'반송파 {FC} Hz, 최대 편이 ±{DELTA_F} Hz → 순간 주파수 범위 {FC-DELTA_F} ~ {FC+DELTA_F} Hz')

### 🐍 파이썬 문법 노트 ① — `import`, 변수, 주석, f-string

**`import`** : 남이 만들어 둔 도구 상자(라이브러리)를 가져옵니다.
```python
import numpy as np            # numpy 전체를 가져와 'np' 라는 별명으로 부른다
from scipy import signal      # scipy 안의 signal 만 꺼내 온다 → signal.butter(...) 로 사용
```
`as` 는 별명입니다. 별명이 없으면 매번 `numpy.cos(...)` 라고 써야 하지만, 별명 덕분에 `np.cos(...)` 로 짧게 씁니다.

**변수** : `이름 = 값` 으로 값에 이름표를 붙입니다. `=` 는 '같다'가 아니라 **'오른쪽 값을 왼쪽 이름에 넣는다'** 는 뜻입니다.
파이썬은 이름을 대문자로 쓰면 상수라고 강제하지 않지만, **'바꾸지 말자'는 프로그래머끼리의 약속** 으로 대문자를 씁니다.

**주석** : `#` 뒤의 내용은 컴퓨터가 무시합니다. 사람을 위한 설명입니다.

**f-string** : 문자열 앞에 `f` 를 붙이면 `{ }` 안에 변수나 계산식을 넣을 수 있습니다.
```python
FC = 10000
print(f'반송파는 {FC} Hz')           # → 반송파는 10000 Hz
print(f'편이 범위 {FC-3000} Hz')     # → { } 안에서 계산도 가능
print(f'{3.14159:.2f}')              # → 3.14  (:.2f = 소수점 둘째 자리까지)
print(f'{48000:,}')                  # → 48,000 (:, = 천 단위 쉼표)
```

> ✏️ **해 보기** : `DELTA_F` 를 5000 으로 바꾸고 셀을 다시 실행하면 마지막 줄 출력이 어떻게 바뀌나요? (확인 후 3000으로 되돌리세요)

## 1. 음성 입력 — 마이크로 녹음하거나 파일 올리기

- `SOURCE = 'record'` : 셀을 실행하면 브라우저가 마이크 권한을 묻습니다 → **허용** → `REC_SEC` 초 동안 말하기
- `SOURCE = 'upload'` : 휴대폰 녹음 파일(m4a, mp3, wav 등 아무 형식)을 올립니다. `START_SEC`, `MAX_SEC` 로 쓸 구간을 고릅니다.

> 🎙️ 녹음 팁 : "하나, 둘, 셋, 에프엠 변조 실험입니다" 처럼 **또박또박, 조금 크게** 말하면 결과가 잘 보입니다.

In [ ]:
SOURCE    = 'record'   # 'record' (마이크 녹음) 또는 'upload' (파일 업로드)
REC_SEC   = 5          # 녹음 시간 [초]
START_SEC = 0          # (업로드) 몇 초 지점부터 사용할지
MAX_SEC   = 10         # (업로드) 최대 몇 초까지 사용할지

# 브라우저에서 실행될 자바스크립트 녹음 코드 (''' ''' 는 여러 줄 문자열)
RECORD_JS = '''
const sleep = t => new Promise(r => setTimeout(r, t));
const toDataURL = blob => new Promise(res => {
  const rd = new FileReader(); rd.onloadend = () => res(rd.result); rd.readAsDataURL(blob); });
var record = async (sec) => {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const rec = new MediaRecorder(stream); const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  const msg = document.createElement('p'); document.body.appendChild(msg);
  rec.start();
  for (let i = sec; i > 0; i--) { msg.innerText = '● 녹음 중... ' + i + '초 남음'; await sleep(1000); }
  const stopped = new Promise(r => rec.onstop = r); rec.stop(); await stopped;
  stream.getTracks().forEach(t => t.stop());
  msg.innerText = '녹음 완료';
  return await toDataURL(new Blob(chunks));
}
'''

def record_voice(sec, fname='voice_raw.webm'):
    # 마이크로 sec 초 녹음해서 fname 파일로 저장하고, 그 파일 이름을 돌려준다.
    display(Javascript(RECORD_JS))                 # ① 브라우저에 녹음 함수 심기
    data = output.eval_js(f'record({sec})')        # ② 녹음 실행 → 'data:audio/webm;base64,....' 문자열
    with open(fname, 'wb') as f:                   # ③ 파일을 '쓰기(w) + 바이너리(b)' 모드로 열기
        f.write(base64.b64decode(data.split(',')[1]))  #    쉼표 뒤의 실제 데이터만 해독해서 저장
    return fname

def load_audio(path, fs=FS, start=0, dur=None):
    # ffmpeg로 어떤 형식이든 '모노 · fs Hz' WAV로 바꾼 뒤 -1 ~ 1 사이 실수 배열로 읽는다.
    cmd = ['ffmpeg', '-y', '-loglevel', 'error', '-ss', str(start), '-i', path]
    if dur:                                        # dur 가 None 이 아니면(=길이를 지정했으면)
        cmd += ['-t', str(dur)]                    #   리스트 뒤에 옵션 두 개를 덧붙인다
    cmd += ['-ac', '1', '-ar', str(fs), 'voice.wav']
    subprocess.run(cmd, check=True)
    _, x = wavfile.read('voice.wav')               # (샘플링주파수, 데이터) 중 데이터만 받기
    return x.astype(np.float32) / 32768.0          # 16비트 정수(-32768~32767) → 실수(-1~1)

if SOURCE == 'record':
    raw = record_voice(REC_SEC)
    x = load_audio(raw)
elif SOURCE == 'upload':
    up = files.upload()                            # 업로드 창 → {파일이름: 내용} 딕셔너리
    raw = list(up.keys())[0]                       # 첫 번째 파일 이름
    x = load_audio(raw, start=START_SEC, dur=MAX_SEC)
else:
    raise ValueError("SOURCE 는 'record' 또는 'upload' 로 써 주세요")

print(f'입력: {raw}  |  길이 {len(x)/FS:.2f}초  |  샘플 {len(x):,}개')
display(Audio(x, rate=FS))

### 🐍 파이썬 문법 노트 ② — 함수 `def`, 기본값 인자, `if / elif / else`, `with`

**함수** : 자주 쓰는 일을 묶어 이름을 붙인 것입니다.
```python
def load_audio(path, fs=FS, start=0, dur=None):   # 함수 정의 (콜론 : 으로 끝남)
    ...                                            # 들여쓰기(공백 4칸)된 부분이 함수 몸통
    return 결과                                     # 결과를 호출한 곳으로 돌려준다
```
- `path` 처럼 기본값이 없는 인자는 **반드시** 넣어야 합니다.
- `fs=FS`, `start=0` 처럼 `=값` 이 붙은 인자는 **기본값 인자** 라서 생략할 수 있습니다.
  `load_audio(raw)` 와 `load_audio(raw, start=3, dur=5)` 둘 다 됩니다. 뒤처럼 `이름=값` 으로 넘기는 것을 **키워드 인자** 라고 합니다.
- `None` 은 '아무것도 없음'을 뜻하는 특별한 값입니다. `if dur:` 는 dur 가 `None`(또는 0)이면 거짓이 됩니다.

**조건문** : 위에서부터 차례로 검사해서 **처음으로 참인 블록 하나만** 실행합니다.
```python
if SOURCE == 'record':     # == 는 '같은가?' 를 묻는 비교 (= 는 대입)
    ...
elif SOURCE == 'upload':   # elif = else if
    ...
else:                      # 위 조건이 모두 거짓일 때
    raise ValueError(...)  # 일부러 오류를 내서 잘못된 설정을 알려 준다
```

**`with open(...) as f:`** : 파일을 열고, 블록이 끝나면 **자동으로 닫아** 줍니다. 닫기를 잊어버릴 걱정이 없습니다.

**리스트 더하기** : `cmd += ['-t', '5']` 는 `cmd = cmd + ['-t', '5']` 와 같고, 리스트 뒤에 항목을 이어 붙입니다.

**문자열 나누기** : `'abc,xyz'.split(',')` → `['abc', 'xyz']`, 그래서 `[1]` 은 쉼표 뒤 부분입니다. (번호는 **0부터** 셉니다)

> ✏️ **해 보기** : `SOURCE = 'radio'` 로 바꾸고 실행하면 어떤 일이 일어나나요?

In [ ]:
# 원본 음성 파형 보기
t_raw = np.arange(len(x)) / FS          # 0, 1/FS, 2/FS, ... 시간 축 [초]

plt.figure(figsize=(11, 2.8))
plt.plot(t_raw, x, 'k', lw=0.4)
plt.title('녹음된 원본 음성')
plt.xlabel('시간 [s]'); plt.ylabel('진폭')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

print(f'가장 큰 값: {np.max(x):.3f},  가장 작은 값: {np.min(x):.3f}')

## 2. 신호파 만들기 — 음성 대역만 남기고 크기 맞추기

FM 변조기에 넣을 신호(= 신호파, 메시지 신호 $x(t)$)를 준비합니다.

1. **대역통과 필터 (300 ~ 3400 Hz)** : 전화처럼 말소리에 꼭 필요한 대역만 남깁니다. 대역을 정해 두어야 FM 대역폭을 계산할 수 있습니다.
2. **정규화** : 가장 큰 값이 정확히 1이 되도록 나눕니다.
   그러면 **목소리가 가장 클 때 반송파 주파수가 정확히 $f_c \pm \Delta f$** 가 됩니다. (|x(t)| ≤ 1 이어야 Δf 의 의미가 정확해집니다)

In [ ]:
# ① 대역통과 필터 설계 (6차 버터워스) → ② 앞뒤로 한 번씩 걸어 위상 지연 없이 필터링
sos_bp = signal.butter(6, [F_LOW, F_HIGH], btype='band', fs=FS, output='sos')
msg = signal.sosfiltfilt(sos_bp, x)

# ③ 정규화 : 배열 전체를 한 번에 나눈다 (NumPy 배열 연산)
msg = msg / np.max(np.abs(msg))

t = np.arange(len(msg)) / FS            # 이후 계속 쓸 시간 축
print(f'신호파 준비 완료: 최댓값 {np.max(msg):.2f}, 최솟값 {np.min(msg):.2f}')

# ④ 가장 소리가 큰 지점 근처 20 ms 를 확대해서 보기
s0 = max(int(np.argmax(np.abs(msg))) - int(0.01*FS), 0)   # 최댓값 위치보다 10 ms 앞
w  = int(0.02 * FS)                                        # 20 ms 는 샘플 몇 개?

fig, ax = plt.subplots(2, 1, figsize=(11, 5.5))
ax[0].plot(t, msg, 'k', lw=0.4)
ax[0].axvspan(t[s0], t[s0+w], color='0.8')                 # 확대 구간 표시
ax[0].set_title('신호파 x(t) — 전체 (회색 띠 = 아래 확대 구간)'); ax[0].set_xlabel('시간 [s]')
ax[1].plot(t[s0:s0+w]*1000, msg[s0:s0+w], 'k', lw=1)      # 슬라이싱으로 일부만 꺼내기
ax[1].set_title('신호파 x(t) — 20 ms 확대'); ax[1].set_xlabel('시간 [ms]')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('▶ 필터링된 신호파 (전화 음질처럼 들림)'); display(Audio(msg, rate=FS))

### 🐍 파이썬 문법 노트 ③ — NumPy 배열 연산과 슬라이싱

**배열 연산** : NumPy 배열은 `for` 문 없이 **모든 원소에 한꺼번에** 계산이 적용됩니다.
```python
a = np.array([2, -4, 1])
np.abs(a)            # [2, 4, 1]     ← 각 원소의 절댓값
np.max(np.abs(a))    # 4             ← 그중 가장 큰 값
a / 4                # [0.5, -1, 0.25] ← 모든 원소를 4로 나눔 (정규화!)
```
`msg = msg / np.max(np.abs(msg))` 는 "절댓값이 가장 큰 값으로 전체를 나눠서 -1 ~ 1 로 맞춘다"는 한 줄입니다.

**슬라이싱** `배열[시작:끝]` : 시작 번호부터 **끝 번호 바로 앞까지** 꺼냅니다.
```python
a = np.array([10, 20, 30, 40, 50])
a[1:3]    # [20, 30]        (1번, 2번 — 3번은 포함 안 됨)
a[:2]     # [10, 20]        (처음부터)
a[3:]     # [40, 50]        (끝까지)
a[-1]     # 50              (음수 = 뒤에서부터)
```
`msg[s0:s0+w]` 는 s0번부터 w개, 즉 **20 ms 분량** 을 꺼냅니다.

**`int(...)`** : `0.02 * 48000 = 960.0` 은 실수인데, 배열 번호는 정수여야 하므로 `int()` 로 바꿉니다.

**`np.argmax(...)`** : 가장 큰 값 **자체** 가 아니라, 그 값이 있는 **위치(번호)** 를 돌려줍니다.

> ✏️ **해 보기** : `w = int(0.02 * FS)` 의 0.02 를 0.005 로 바꾸면 그래프가 어떻게 변하나요?

## 3. FM 변조 — 목소리로 반송파의 '빠르기'를 바꾸기

### 3-1. 핵심 아이디어
FM 에서는 **순간 주파수** 가 목소리를 따라 움직입니다.

$$f_i(t) = f_c + \Delta f \cdot x(t)$$

- 목소리가 0 일 때 → 반송파는 정확히 $f_c$ = 10000 Hz
- 목소리가 +1 (최대) 일 때 → $f_c + \Delta f$ = 13000 Hz (더 촘촘)
- 목소리가 −1 (최소) 일 때 → $f_c - \Delta f$ = 7000 Hz (더 성김)

그런데 코사인에 넣는 것은 주파수가 아니라 **위상(각도)** 입니다. 위상은 주파수를 **시간에 대해 쌓은 것(적분)** 이므로

$$s(t) = A_c \cos\!\Big(2\pi f_c t + 2\pi \Delta f \int_0^t x(\tau)\,d\tau\Big)$$

### 3-2. 컴퓨터로 적분하기 — 누적합 `np.cumsum`
적분 = 아주 짧은 시간 $dt$ 동안의 넓이 $x \cdot dt$ 를 계속 더해 가는 것입니다.
```python
np.cumsum([1, 2, 3, 4])   # [1, 3, 6, 10]  ← 지금까지의 합계를 차례로
```
그래서 $\int x\,dt \approx$ `np.cumsum(msg) / FS` 입니다. ($dt = 1/FS$)

In [ ]:
Ac = 1.0                                # 반송파 진폭 (FM 에서는 항상 일정!)
dt = 1 / FS                             # 샘플 사이 시간 간격 [s]

integral_msg = np.cumsum(msg) * dt                       # ① 신호파 적분 (누적합)
phase        = 2*np.pi*FC*t + 2*np.pi*DELTA_F*integral_msg  # ② 위상 = 반송파 위상 + 음성으로 생긴 위상
fm_sig       = Ac * np.cos(phase)                         # ③ FM 신호

inst_freq_tx = FC + DELTA_F * msg                         # (확인용) 송신 측 순간 주파수

print(f'FM 신호 생성 완료: 샘플 {len(fm_sig):,}개, 진폭 범위 {fm_sig.min():.2f} ~ {fm_sig.max():.2f}')
print(f'순간 주파수 범위: {inst_freq_tx.min():.0f} ~ {inst_freq_tx.max():.0f} Hz')

In [ ]:
# 같은 20 ms 구간에서 신호파 · 순간 주파수 · FM 파형을 위아래로 비교
tt = t[s0:s0+w] * 1000                  # 확대 구간의 시간 [ms]

fig, ax = plt.subplots(3, 1, figsize=(11, 8), sharex=True)
ax[0].plot(tt, msg[s0:s0+w], 'k', lw=1)
ax[0].set_title('① 신호파 x(t)')

ax[1].plot(tt, inst_freq_tx[s0:s0+w], 'k', lw=1)
ax[1].axhline(FC, color='0.6', ls='--', lw=1, label=f'반송파 {FC} Hz')
ax[1].set_title('② 순간 주파수  f_c + Δf·x(t)  — 신호파와 모양이 똑같다!'); ax[1].set_ylabel('Hz')
ax[1].legend(loc='upper right')

z = int(0.004 * FS)                     # FM 파형은 너무 촘촘하니 앞 4 ms 만
ax[2].plot(tt[:z], fm_sig[s0:s0+z], 'k', lw=0.8)
ax[2].set_title('③ FM 신호 (앞 4 ms) — 크기는 일정, 간격만 변한다'); ax[2].set_xlabel('시간 [ms]')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

> 🔍 **관찰** : ③ 그래프에서 파형의 **높이는 전혀 변하지 않습니다.** 정보는 오직 파형의 간격(주파수)에만 들어 있습니다.
> AM 실습의 변조 파형(위아래가 부풀었다 줄었다)과 비교해 보세요.

### 3-3. 변조지수 β 와 카슨의 법칙

- **변조지수** $\beta = \dfrac{\Delta f}{f_m}$ ($f_m$ = 신호파의 최고 주파수)
- **카슨의 법칙** : FM 신호가 차지하는 대역폭 $B \approx 2(\Delta f + f_m)$

음성은 최고 주파수가 $f_m$ = F_HIGH = 3400 Hz 이므로, 아래 셀에서 계산한 이론 대역폭과 **스펙트럼에서 실제로 잰 대역폭** 을 비교해 봅니다.

In [ ]:
def spectrum_db(sig, fs=FS):
    # 웰치 방법으로 전력 스펙트럼을 구해 (주파수 배열, dB 배열) 두 개를 돌려준다.
    f, p = signal.welch(sig, fs=fs, nperseg=4096)
    return f, 10*np.log10(p + 1e-20), p     # 1e-20 : log(0) 오류 방지용 아주 작은 수

def occupied_bw(f, p, ratio=0.99):
    # 전체 전력의 ratio(99%)가 들어 있는 주파수 구간 [f_lo, f_hi] 를 찾는다.
    cum = np.cumsum(p) / np.sum(p)          # 누적 전력 비율 0 → 1
    lo_mask = cum >= (1 - ratio) / 2        # 불리언 마스크: 조건을 만족하면 True
    hi_mask = cum >= 1 - (1 - ratio) / 2
    return f[lo_mask][0], f[hi_mask][0]     # 처음으로 True 가 되는 주파수

beta     = DELTA_F / F_HIGH
B_carson = 2 * (DELTA_F + F_HIGH)

f, msg_db, _     = spectrum_db(msg)         # 반환값 3개 중 필요 없는 건 _ 로 받기
f, fm_db, fm_p   = spectrum_db(fm_sig)
f_lo, f_hi       = occupied_bw(f, fm_p)

print(f'변조지수 β = Δf / f_m = {DELTA_F} / {F_HIGH} = {beta:.2f}')
print(f'카슨 대역폭 B ≈ 2 × ({DELTA_F} + {F_HIGH}) = {B_carson} Hz  →  {FC - B_carson//2} ~ {FC + B_carson//2} Hz')
print(f'실제 측정(전력 99%): {f_lo:.0f} ~ {f_hi:.0f} Hz  →  폭 {f_hi - f_lo:.0f} Hz')

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(f, msg_db, 'k', lw=1); ax[0].set_title('신호파(음성) 스펙트럼 — 300~3400 Hz 에 모여 있음')
ax[1].plot(f, fm_db, 'k', lw=1)
ax[1].axvspan(FC - B_carson/2, FC + B_carson/2, color='0.85', label='카슨 대역폭')
ax[1].axvline(FC, color='0.5', ls='--', lw=1, label=f'반송파 {FC} Hz')
ax[1].set_title('FM 신호 스펙트럼 — 반송파 주변으로 넓게 퍼짐'); ax[1].set_xlabel('주파수 [Hz]')
ax[1].legend(loc='upper right')
for a in ax:
    a.set_ylabel('dB'); a.set_xlim(0, 20000); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 🐍 파이썬 문법 노트 ④ — `return` 여러 개, `_`, 불리언 마스크

**여러 값 돌려주기 / 받기 (튜플 언패킹)**
```python
def spectrum_db(sig):
    ...
    return f, db, p          # 쉼표로 여러 개를 한 번에 돌려준다

f, fm_db, fm_p = spectrum_db(fm_sig)   # 왼쪽에 같은 개수의 이름을 쓰면 순서대로 받는다
f, msg_db, _   = spectrum_db(msg)      # 안 쓸 값은 _ 로 받는 것이 관례
```

**불리언 마스크** : 비교 연산을 배열에 쓰면 True/False 배열이 나오고, 그것으로 원하는 원소만 골라낼 수 있습니다.
```python
cum = np.array([0.001, 0.2, 0.6, 0.996, 1.0])
f   = np.array([100,   200, 300, 400,   500])
mask = cum >= 0.995        # [False, False, False, True, True]
f[mask]                    # [400, 500]   ← True 인 자리만
f[mask][0]                 # 400          ← 그중 첫 번째
```

**`//` (몫 나눗셈)** : `12800 // 2` → `6400` (정수 결과). `/` 는 항상 실수 결과(`6400.0`)를 냅니다.

> ✏️ **해 보기** : 카슨 대역폭과 실제 99% 대역폭은 얼마나 비슷한가요? 실제 음성으로 재면 카슨 값보다 꽤 **작게** 나옵니다. 사람 목소리는 대부분의 시간 동안 최대 크기가 아니고, 에너지도 3400 Hz 가 아니라 낮은 주파수(수백 Hz)에 몰려 있기 때문입니다. 카슨의 법칙은 '최악의 경우'를 대비한 넉넉한 어림값이라는 뜻이죠.

## 4. 송신 — FM 신호를 들어 보고, 잡음 채널을 통과시키기

FM 신호는 10 kHz 근처의 소리이므로 귀로 들을 수 있습니다. 말을 하는 순간마다 **'삐' 소리의 높이가 흔들리는 것** 을 들어 보세요.
(AM 신호는 소리의 **크기** 가 흔들렸습니다)

그다음 실제 무선 채널처럼 **가우시안 잡음(AWGN)** 을 더합니다. 잡음의 크기는 SNR 로 정합니다.

$$\text{SNR}_{dB} = 10\log_{10}\frac{P_{신호}}{P_{잡음}} \quad\Rightarrow\quad P_{잡음} = \frac{P_{신호}}{10^{\text{SNR}_{dB}/10}}$$

In [ ]:
# ① FM 신호를 WAV 파일로 저장 (실수 -1~1 → 16비트 정수)
wavfile.write('fm_tx.wav', FS, (fm_sig * 32767 * 0.9).astype(np.int16))
print('▶ 송신 FM 신호 (fm_tx.wav 로 저장됨 — 왼쪽 📁 파일 탭에서 내려받기 가능)')
display(Audio(fm_sig, rate=FS))

# ② 잡음 채널 함수
rng = np.random.default_rng(seed=0)     # 난수 발생기 (seed 를 고정하면 매번 같은 잡음)

def add_noise(sig, snr_db):
    '''신호에 snr_db [dB] 가 되도록 가우시안 잡음을 더해서 돌려준다.'''
    p_sig   = np.mean(sig**2)                       # 신호 전력 = 제곱의 평균
    p_noise = p_sig / 10**(snr_db / 10)             # ** 는 거듭제곱
    noise   = rng.standard_normal(len(sig)) * np.sqrt(p_noise)
    return sig + noise

rx = add_noise(fm_sig, SNR_DB)
print(f'▶ 채널 통과 후 수신 신호 (SNR {SNR_DB} dB)')
display(Audio(rx, rate=FS))

z = int(0.004 * FS)
plt.figure(figsize=(11, 3))
plt.plot(t[s0:s0+z]*1000, rx[s0:s0+z], color='0.6', lw=0.8, label='수신 신호 (잡음 포함)')
plt.plot(t[s0:s0+z]*1000, fm_sig[s0:s0+z], 'k', lw=1, label='송신 FM 신호')
plt.title(f'잡음 채널 통과 (SNR = {SNR_DB} dB, 4 ms 확대)'); plt.xlabel('시간 [ms]')
plt.legend(loc='upper right'); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

### 🐍 파이썬 문법 노트 ⑤ — 독스트링, 거듭제곱, 자료형 바꾸기

**독스트링** : 함수 바로 첫 줄에 `'''설명'''` 을 쓰면 그 함수의 공식 설명서가 됩니다.
`help(add_noise)` 를 실행하면 이 설명이 출력됩니다. (아래 빈 셀에서 직접 해 보세요)

**연산자**
| 기호 | 뜻 | 예 |
|---|---|---|
| `**` | 거듭제곱 | `10**2` → 100,  `sig**2` → 배열의 모든 원소를 제곱 |
| `/` | 나눗셈 | `7/2` → 3.5 |
| `//` | 몫 | `7//2` → 3 |
| `%` | 나머지 | `7%2` → 1 |

**`.astype(np.int16)`** : 배열의 자료형을 바꿉니다. WAV 파일은 보통 16비트 정수(−32768 ~ 32767)로 저장하므로 32767 을 곱한 뒤 정수로 바꿉니다. 0.9 를 곱한 것은 끝값에서 소리가 찌그러지지 않게 여유를 둔 것입니다.

> ✏️ **해 보기** : 맨 위 0단계의 `SNR_DB` 를 0 으로 바꾸고 여기까지 다시 실행해 보세요. 잡음이 들리지만 '삐' 소리의 높이 변화는 여전히 느껴지나요?

In [ ]:
help(add_noise)

## 5. 수신 · 복조 — 순간 주파수를 재서 목소리 되찾기

FM 수신기는 "지금 이 순간 반송파가 몇 Hz 인가?" 를 계속 재면 됩니다. 그 값에서 $f_c$ 를 빼면 $\Delta f \cdot x(t)$, 즉 목소리가 나옵니다.

| 단계 | 하는 일 | 코드 |
|---|---|---|
| ① 대역통과 | 카슨 대역 밖 잡음을 미리 잘라 냄 | `signal.sosfiltfilt` |
| ② 해석 신호 | 힐베르트 변환으로 신호의 **위상(각도)** 을 꺼냄 | `signal.hilbert`, `np.angle` |
| ③ 위상 펴기 | −π~π 에서 끊기는 각도를 이어 붙임 | `np.unwrap` |
| ④ 미분 | 위상의 변화율 = 주파수 (3단계 적분의 **반대**) | `np.diff(...) * FS / (2π)` |
| ⑤ 반송파 빼기 | $f_i - f_c = \Delta f\, x(t)$ | `- FC` |
| ⑥ 저역통과 | 음성 대역(3400 Hz) 위 잡음 제거 | `signal.sosfiltfilt` |

> 💡 **AM 과의 차이** : AM 은 '크기(포락선)'를 재고, FM 은 '각도의 변화 속도'를 잽니다. 잡음은 주로 **크기** 를 흔들기 때문에 FM 이 잡음에 강합니다.

> 💡 **확장 실험 (음향 채널)** : 다른 휴대폰으로 4단계의 송신 소리를 스피커로 틀어 **녹음** 한 뒤 `RX_FROM_FILE = True` 로 업로드해 보세요. 실제 공기를 거친 FM 신호를 복조합니다. FM 은 스피커/마이크의 음량 차이에 거의 영향을 받지 않습니다.

In [ ]:
RX_FROM_FILE = False      # True : 녹음해 온 수신 파일을 업로드해서 복조

if RX_FROM_FILE:
    up = files.upload()
    rx = load_audio(list(up.keys())[0])

def fm_demod(sig, fs=FS, fc=FC, df=DELTA_F, cutoff=F_HIGH):
    '''FM 신호 sig 를 복조한다.
    반환: (복원 음성 -1~1, 측정한 순간 주파수 [Hz])'''
    # ① 카슨 대역 밖 잡음 제거
    half_bw = df + cutoff + 500                              # 여유 500 Hz
    band = [max(fc - half_bw, 100), min(fc + half_bw, fs/2 - 100)]
    sos_rx = signal.butter(6, band, btype='band', fs=fs, output='sos')
    sig = signal.sosfiltfilt(sos_rx, sig)

    # ② ~ ④ 위상 → 순간 주파수
    analytic  = signal.hilbert(sig)                          # 복소수 해석 신호
    ph        = np.unwrap(np.angle(analytic))                # 이어 붙인 위상 [rad]
    inst_freq = np.diff(ph) * fs / (2*np.pi)                 # 위상 미분 → 주파수 [Hz]
    inst_freq = np.append(inst_freq, inst_freq[-1])          # diff 로 하나 줄어든 길이 맞추기

    # ⑤ 반송파 빼고 Δf 로 나누기 → x(t)
    out = (inst_freq - fc) / df

    # ⑥ 음성 대역 저역통과 + 직류 제거 + 정규화
    sos_lp = signal.butter(6, cutoff, btype='low', fs=fs, output='sos')
    out = signal.sosfiltfilt(sos_lp, out)
    out = out - np.mean(out)
    return out / (np.max(np.abs(out)) + 1e-12), inst_freq

recovered, inst_freq_rx = fm_demod(rx)

In [ ]:
n = min(len(msg), len(recovered))       # 길이가 다를 수 있으니 짧은 쪽에 맞추기

fig, ax = plt.subplots(2, 1, figsize=(11, 6.5))
ax[0].plot(t[s0:s0+w]*1000, inst_freq_rx[s0:s0+w], color='0.65', lw=0.6, label='수신 측에서 잰 순간 주파수 (필터 전)')
ax[0].plot(t[s0:s0+w]*1000, inst_freq_tx[s0:s0+w], 'k', lw=1.2, label='송신 측 순간 주파수')
ax[0].set_ylim(FC - 2*DELTA_F, FC + 2*DELTA_F)
ax[0].set_title('순간 주파수 측정 (20 ms 확대)'); ax[0].set_xlabel('시간 [ms]'); ax[0].set_ylabel('Hz')
ax[0].legend(loc='upper right')

ax[1].plot(t[:n], msg[:n], color='0.65', lw=0.6, label='원래 신호파')
ax[1].plot(t[:n], recovered[:n], 'k', lw=0.4, ls='--', label='복원된 음성')
ax[1].set_title('원래 신호 vs 복원 신호'); ax[1].set_xlabel('시간 [s]')
ax[1].legend(loc='upper right')
for a in ax: a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

if not RX_FROM_FILE:                    # 직접 만든 신호일 때만 시간이 정확히 맞으므로 비교 가능
    corr = np.corrcoef(msg[:n], recovered[:n])[0, 1]
    print(f'원래 신호와 복원 신호의 상관계수: {corr:.3f}  (1에 가까울수록 잘 복원)')

print('▶ 복원된 음성'); display(Audio(recovered, rate=FS))
wavfile.write('fm_recovered.wav', FS, (recovered * 32767 * 0.9).astype(np.int16))

### 🐍 파이썬 문법 노트 ⑥ — 리스트 인덱스, `not`, `min/max`

- `inst_freq[-1]` : 배열의 **마지막** 원소. `np.append(배열, 값)` 은 뒤에 값을 하나 덧붙인 **새 배열** 을 돌려줍니다.
- `np.diff([1, 4, 9, 16])` → `[3, 5, 7]` : 이웃한 값의 차이. 결과가 **하나 짧아지므로** 길이를 맞춰 준 것입니다. (`cumsum` ↔ `diff` 는 적분 ↔ 미분 관계!)
- `max(a, 100)` 은 둘 중 큰 값, `min(a, b)` 는 작은 값 → 필터 주파수가 0 아래나 FS/2 위로 넘어가지 않게 막는 **안전장치** 입니다.
- `if not RX_FROM_FILE:` : `not` 은 참/거짓을 뒤집습니다. "파일에서 받은 게 **아니면**" 이라는 뜻입니다.
- `np.corrcoef(a, b)` 는 2×2 표를 돌려주고, `[0, 1]` 은 그 표의 0행 1열 = a 와 b 의 상관계수입니다.

## 6. 실험

### 실험 ① — 주파수 편이 Δf 를 바꾸면?

Δf 를 여러 값으로 바꿔 가며 **대역폭** 과 **잡음 속 복원 품질** 이 어떻게 변하는지 `for` 문으로 한 번에 비교합니다.
(잡음은 일부러 크게, SNR = 10 dB 로 둡니다)

In [ ]:
df_list  = [500, 1500, 3000, 5000]      # 실험할 Δf 값들 (리스트)
TEST_SNR = 10
results  = []                           # 결과를 모아 둘 빈 리스트

fig, ax = plt.subplots(len(df_list), 1, figsize=(11, 9), sharex=True)

for i, df in enumerate(df_list):        # i = 0,1,2,3  /  df = 500, 1500, ...
    s = np.cos(2*np.pi*FC*t + 2*np.pi*df*np.cumsum(msg)/FS)   # 같은 음성, 다른 Δf 로 변조
    r, _ = fm_demod(add_noise(s, TEST_SNR), df=df)            # 잡음 채널 → 복조
    c = np.corrcoef(msg, r)[0, 1]

    f, s_db, _ = spectrum_db(s)
    ax[i].plot(f, s_db, 'k', lw=1)
    ax[i].set_title(f'Δf = {df} Hz   (β = {df/F_HIGH:.2f},  카슨 B = {2*(df+F_HIGH)} Hz,  상관계수 {c:.3f})')
    ax[i].set_xlim(0, 20000); ax[i].set_ylabel('dB'); ax[i].grid(alpha=0.3)

    results.append({'df': df, 'beta': df/F_HIGH, 'B': 2*(df+F_HIGH), 'corr': c, 'audio': r})

ax[-1].set_xlabel('주파수 [Hz]')
plt.tight_layout(); plt.show()

# 결과 표 출력 (f-string 의 칸 맞추기 :>6 = 오른쪽 정렬 6칸)
print(f"{'Δf[Hz]':>7} | {'β':>5} | {'대역폭[Hz]':>9} | {'상관계수':>6}")
print('-' * 42)
for res in results:
    print(f"{res['df']:>7} | {res['beta']:>5.2f} | {res['B']:>11} | {res['corr']:>8.3f}")

In [ ]:
# Δf 별로 복원된 음성 들어 보기
for res in results:
    print(f"▶ Δf = {res['df']} Hz  (상관계수 {res['corr']:.3f})")
    display(Audio(res['audio'], rate=FS))

> 🔍 **관찰** : Δf 를 키우면 스펙트럼이 넓어지는 대신(= 대역폭을 더 씀) 잡음 속에서도 음성이 더 깨끗하게 복원됩니다.
> **대역폭을 내주고 잡음 내성을 사는 것** — 이것이 FM 의 핵심 거래(trade-off)입니다.
> 실제 FM 방송은 Δf = 75 kHz 로, 음성 대역(15 kHz)보다 훨씬 크게 씁니다.

### 🐍 파이썬 문법 노트 ⑦ — `for`, `enumerate`, 리스트, 딕셔너리

**`for` 반복문** : 리스트의 원소를 하나씩 꺼내 같은 일을 반복합니다.
```python
for df in [500, 1500, 3000]:
    print(df)            # 500 → 1500 → 3000 차례로 출력
```

**`enumerate`** : 원소와 함께 **번호(0부터)** 도 같이 줍니다. 그래프 칸 `ax[i]` 를 고를 때 번호가 필요해서 썼습니다.
```python
for i, df in enumerate([500, 1500]):
    print(i, df)         # 0 500 → 1 1500
```

**리스트 `[ ]`** : 순서가 있는 묶음. `results.append(값)` 으로 뒤에 추가합니다. `ax[-1]` 은 마지막 그래프 칸입니다.

**딕셔너리 `{ }`** : **이름(키) → 값** 으로 저장하는 묶음. 번호 대신 이름으로 꺼냅니다.
```python
res = {'df': 3000, 'corr': 0.91}
res['df']        # 3000
res['corr']      # 0.91
```

**f-string 칸 맞추기** : `{값:>7}` 은 7칸 오른쪽 정렬, `{값:>8.3f}` 는 8칸 오른쪽 정렬 + 소수 셋째 자리. 표를 깔끔하게 출력할 때 씁니다.
f-string 안에서 딕셔너리 키를 쓸 때는 바깥 따옴표(`"`)와 안쪽 따옴표(`'`)를 다르게 써야 합니다.

> ✏️ **해 보기** : `df_list` 에 `8000` 을 추가해 보세요. 카슨 대역폭이 반송파 아래쪽(0 Hz)에 닿지 않는지 확인하세요. 만약 `FC - B/2 < 0` 이 되면 어떤 문제가 생길까요?

### 실험 ② — 같은 잡음에서 AM 과 FM 비교

같은 음성, 같은 반송파(10 kHz), 같은 SNR 에서 AM 과 FM 중 어느 쪽이 음성을 더 잘 복원하는지 비교합니다.
AM 은 AM 실습과 같은 방식(변조도 m = 0.8, 포락선 검파)을 사용합니다.

In [ ]:
M = 0.8                                                    # AM 변조도

def am_modulate(x, m=M, fc=FC, fs=FS):
    '''AM 변조: (1 + m·x) · cos(2π fc t), 최대 크기를 1로 맞춤'''
    tt = np.arange(len(x)) / fs
    s = (1 + m*x) * np.cos(2*np.pi*fc*tt)
    return s / np.max(np.abs(s))

def am_demod(sig, fs=FS, cutoff=F_HIGH):
    '''AM 포락선 검파: 정류 → 저역통과 → 직류 제거'''
    sos_lp = signal.butter(6, cutoff, btype='low', fs=fs, output='sos')
    env = signal.sosfiltfilt(sos_lp, np.abs(sig))
    out = env - np.mean(env)
    return out / (np.max(np.abs(out)) + 1e-12)

am_sig   = am_modulate(msg)
snr_list = [30, 20, 15, 10, 5, 0]

# 리스트 컴프리헨션: SNR 마다 [잡음 추가 → 복조 → 상관계수] 를 한 줄로
am_corr = [np.corrcoef(msg, am_demod(add_noise(am_sig, s)))[0, 1]    for s in snr_list]
fm_corr = [np.corrcoef(msg, fm_demod(add_noise(fm_sig, s))[0])[0, 1] for s in snr_list]

print(f"{'SNR[dB]':>8} | {'AM':>6} | {'FM':>6} | 더 좋은 쪽")
print('-' * 38)
for s, a, f_ in zip(snr_list, am_corr, fm_corr):           # 세 리스트를 나란히 꺼내기
    better = 'FM' if f_ > a else 'AM'                      # 한 줄 if (조건 표현식)
    print(f'{s:>8} | {a:>6.3f} | {f_:>6.3f} | {better}')

plt.figure(figsize=(8, 4))
plt.plot(snr_list, am_corr, 'o--', color='0.55', label='AM (포락선 검파)')
plt.plot(snr_list, fm_corr, 's-',  color='k',    label=f'FM (Δf = {DELTA_F} Hz)')
plt.gca().invert_xaxis()                                   # 왼쪽이 깨끗, 오른쪽이 잡음 많음
plt.title('채널 잡음에 따른 복원 품질'); plt.xlabel('SNR [dB]  (오른쪽으로 갈수록 잡음 ↑)')
plt.ylabel('상관계수'); plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# 잡음이 큰 채널에서 직접 들어 보고 비교하기
LISTEN_SNR = 5

print(f'▶ AM 복원 음성 (SNR {LISTEN_SNR} dB)')
display(Audio(am_demod(add_noise(am_sig, LISTEN_SNR)), rate=FS))
print(f'▶ FM 복원 음성 (SNR {LISTEN_SNR} dB)')
display(Audio(fm_demod(add_noise(fm_sig, LISTEN_SNR))[0], rate=FS))

> 🔍 **관찰** : 잡음이 어느 정도 있는 구간(대략 5 ~ 15 dB)에서는 FM 이 AM 보다 깨끗합니다.
> 하지만 잡음이 **아주 크면(0 dB 근처)** FM 도 갑자기 무너집니다. 이것을 FM 의 **문턱 효과(threshold effect)** 라고 합니다.
> 표의 숫자와 귀로 들은 느낌이 일치하는지 확인해 보세요.

### 🐍 파이썬 문법 노트 ⑧ — 리스트 컴프리헨션, `zip`, 한 줄 `if`

**리스트 컴프리헨션** : `for` 문으로 리스트를 만드는 일을 한 줄로 씁니다.
```python
# 일반 for 문
squares = []
for k in [1, 2, 3]:
    squares.append(k**2)

# 리스트 컴프리헨션 (결과는 같다: [1, 4, 9])
squares = [k**2 for k in [1, 2, 3]]
```

**`zip`** : 여러 리스트를 **같은 순서끼리 짝지어** 꺼냅니다.
```python
for s, a in zip([30, 20], [0.99, 0.98]):
    print(s, a)          # 30 0.99 → 20 0.98
```

**조건 표현식 (한 줄 if)** : `값1 if 조건 else 값2` — 조건이 참이면 값1, 거짓이면 값2.
```python
better = 'FM' if f_ > a else 'AM'
```

**`fm_demod(...)[0]`** : fm_demod 는 값 2개(튜플)를 돌려주므로, `[0]` 으로 첫 번째(복원 음성)만 꺼냈습니다.

> ✏️ **해 보기** : 0단계에서 `DELTA_F` 를 5000 으로 바꾸고 전체를 다시 실행(`런타임 → 모두 실행`)하면 실험 ②의 FM 곡선이 어떻게 바뀌나요?

## 7. 생각해 볼 문제

1. FM 변조 파형은 진폭이 항상 일정합니다. 그렇다면 수신기에서 진폭에 생긴 잡음을 어떻게 '무시'할 수 있는 걸까요? (5단계의 복조 과정 중 어느 단계와 관련이 있나요?)
2. 3단계에서 **적분(cumsum)** 을 했고, 5단계에서 **미분(diff)** 을 했습니다. 둘의 관계를 설명해 보세요.
3. Δf 를 키우면 잡음에 강해집니다. 그러면 Δf 를 무한히 키우지 않는 이유는 무엇일까요? (힌트: 카슨의 법칙, 다른 방송국)
4. FC = 10000 Hz, DELTA_F = 8000 Hz 로 바꾸면 카슨 대역폭의 아래쪽 끝은 몇 Hz 인가요? 이때 어떤 문제가 생길까요? 직접 바꿔서 확인해 보세요.
5. 실험 ②에서 SNR 이 0 dB 근처일 때 FM 이 오히려 AM 보다 나빠지기도 합니다. 이유를 '문턱 효과'로 설명해 보세요.
6. 음향 채널 실험(RX_FROM_FILE)에서 녹음한 휴대폰의 음량을 크게/작게 바꿔도 FM 복원 결과는 거의 같습니다. AM 이었다면 어땠을까요?

## 🐍 이번 실습에서 사용한 파이썬 문법 정리

| 문법 | 예 (이 노트북에서) | 뜻 |
|---|---|---|
| `import ... as` | `import numpy as np` | 라이브러리를 별명으로 가져오기 |
| `from ... import` | `from scipy import signal` | 일부만 꺼내 오기 |
| 변수 / 상수 | `FC = 10000` | 값에 이름 붙이기 (대문자 = 바꾸지 말자는 약속) |
| f-string | `f'{beta:.2f}'`, `f'{n:,}'`, `f'{x:>7}'` | 문자열에 값 넣기, 자릿수·정렬 |
| `def` / `return` | `def add_noise(sig, snr_db): ... return` | 함수 만들기 |
| 기본값 · 키워드 인자 | `load_audio(raw, start=3, dur=5)` | 생략 가능한 인자, 이름으로 넘기기 |
| 여러 값 반환 | `return out, inst_freq` → `a, b = f()` | 튜플 언패킹, 안 쓰는 값은 `_` |
| 독스트링 | `'''설명'''`, `help(함수)` | 함수 설명서 |
| `if / elif / else` | `if SOURCE == 'record':` | 조건에 따라 다르게 실행 |
| 한 줄 if | `'FM' if f_ > a else 'AM'` | 조건에 따라 값 고르기 |
| `not`, `None` | `if not RX_FROM_FILE:`, `dur=None` | 참/거짓 뒤집기, '값 없음' |
| `with open` | `with open(fname, 'wb') as f:` | 파일 열고 자동으로 닫기 |
| 리스트 | `df_list = [500, 1500]`, `.append()`, `[-1]` | 순서 있는 묶음 |
| 딕셔너리 | `{'df': 3000}`, `res['df']` | 이름 → 값 묶음 |
| `for` · `enumerate` · `zip` | `for i, df in enumerate(df_list):` | 반복, 번호와 함께, 짝지어 반복 |
| 리스트 컴프리헨션 | `[f(s) for s in snr_list]` | 한 줄로 리스트 만들기 |
| NumPy 배열 연산 | `msg / np.max(np.abs(msg))` | 모든 원소에 한꺼번에 계산 |
| 슬라이싱 | `msg[s0:s0+w]` | 배열 일부 꺼내기 (끝 번호는 미포함) |
| 불리언 마스크 | `f[cum >= 0.995][0]` | 조건을 만족하는 원소만 고르기 |
| 누적합 · 차분 | `np.cumsum` / `np.diff` | 적분 / 미분의 컴퓨터 버전 |
| 연산자 | `**`, `//`, `%` | 거듭제곱, 몫, 나머지 |